In [ ]:
%reload_ext autoreload
%autoreload 2
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from collections import OrderedDict
from qiskit_metal.qlibrary.core import QComponent

from qiskit_metal.qlibrary.sample_shapes.rectangle import Rectangle
from qiskit_metal.qlibrary.sample_shapes.rectangle_hollow import RectangleHollow
from qiskit_metal.qlibrary.user_components.self_define import Cross
from qiskit_metal.qlibrary.user_components.fillet_q import Fillet_Qubit
from qiskit_metal.qlibrary.user_components.round_tee import Round_Tee
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket

from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround

from qiskit_metal.qlibrary.couplers.line_tee import LineTee
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.launchpad_wb_coupled import LaunchpadWirebondCoupled
from qiskit_metal.qlibrary.terminations.launchpad_wb_driven import LaunchpadWirebondDriven

from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.toolbox_metal.parsing import parse_value

from qiskit_metal.analyses.quantization import EPRanalysis
import pyEPR as epr
from qiskit_metal.analyses.simulation import ScatteringImpedanceSim
from qiskit_metal.analyses.sweep_and_optimize.sweeping import Sweeping
from qiskit_metal.qlibrary.core import QComponent

import numpy as np 
import matplotlib.pyplot as plt

In [ ]:
class VirtualJunction(QComponent):
    """
    A component with no geometry, only pins.
    Used to create a T-junction for routing.
    """
    # 定義元件的預設參數
    default_options = dict(
        pin_width='10um' # 讓 pin 的寬度可以被設定
    )

    def make(self):
        """
        This is the method that Qiskit Metal calls to draw the component.
        We will only add pins here and no geometry.
        """
        p = self.p  # 取得解析後的參數 (parsed options)

        # 取得元件的中心位置
        pos_x = p.pos_x
        pos_y = p.pos_y

        # 定義三個 pin，它們的向量都從中心點向外指
        # pin 的格式是 [起點, 終點]
        pin_len = 1e-6 # 一個極小值，只用來定義方向
        
        # pin 的法向量 (normal vector) 會從第一個點指向第二個點
        # Route* 元件會從 pin 的終點開始連接
        pin_left = [[pos_x, pos_y+p.pin_width/2], [pos_x - pin_len, pos_y+p.pin_width/2]]
        pin_right = [[pos_x, pos_y-p.pin_width/2], [pos_x + pin_len, pos_y-p.pin_width/2]]
        pin_bottom = [[pos_x-p.pin_width/2, pos_y], [pos_x-p.pin_width/2, pos_y - pin_len]]
        pin_top = [[pos_x+p.pin_width/2, pos_y], [pos_x+p.pin_width/2, pos_y + pin_len]]

        # 將 pins 加入到元件中
        self.add_pin('left', points=pin_left, width=p.pin_width)
        self.add_pin('right', points=pin_right, width=p.pin_width)
        self.add_pin('bottom', points=pin_bottom, width=p.pin_width)
        self.add_pin('top', points=pin_top, width=p.pin_width)


# Circuit Design

In [ ]:
design = designs.DesignPlanar()
gui = MetalGUI(design)
design.overwrite_enabled = True
design.chips.main.size.size_x = '10 mm'
design.chips.main.size.size_y = '10 mm'
design.chips.main.size.size_z = '-650 um'
design.chips.main.material = 'sapphire'
var = design.variables
print(var)

In [ ]:
"""
Set 8 ports. 2 for readout TL, 3 for charge line and 3 for flux line. 
"""
x_port, y_port = 3.000, 3.000
TL_width, TL_gap = 0.020, 0.012

port_L1 = LaunchpadWirebond(design, 'port_L1', options = dict(pos_x = -x_port, pos_y =  y_port, orientation = '  0', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.020, trace_gap=0.012))
port_L2 = LaunchpadWirebond(design, 'port_L2', options = dict(pos_x = -x_port, pos_y =  1.0, orientation = '  0', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.010, trace_gap=0.006))
port_L3 = LaunchpadWirebond(design, 'port_L3', options = dict(pos_x = -x_port, pos_y = -1.0, orientation = '  0', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.010, trace_gap=0.006))
port_L4 = LaunchpadWirebond(design, 'port_L4', options = dict(pos_x = -x_port, pos_y = -y_port, orientation = '  0', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.010, trace_gap=0.006))
port_R1 = LaunchpadWirebond(design, 'port_R1', options = dict(pos_x =  x_port, pos_y =  y_port, orientation = '180', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.020, trace_gap=0.012))
port_R2 = LaunchpadWirebond(design, 'port_R2', options = dict(pos_x =  x_port, pos_y =  1.0, orientation = '180', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.010, trace_gap=0.006))
port_R3 = LaunchpadWirebond(design, 'port_R3', options = dict(pos_x =  x_port, pos_y = -1.0, orientation = '180', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.010, trace_gap=0.006))
port_R4 = LaunchpadWirebond(design, 'port_R4', options = dict(pos_x =  x_port, pos_y = -y_port, orientation = '180', 
                                                              pad_width = 0.5, pad_height = 0.25, pad_gap = 0.2, taper_height = 0.5,
                                                              lead_length = 0.100, trace_width=0.010, trace_gap=0.006))
gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
"""
Set transmission line (TL).
"""
TL_width, TL_gap = 0.020, 0.012

TL = RouteStraight(design, 'TL', 
    dict(pin_inputs = dict(start_pin = dict(component = 'port_L1', pin = 'tie'), 
                             end_pin = dict(component = 'port_R1', pin = 'tie')),
         trace_width = TL_width, trace_gap = TL_gap, fillet = 0.100, hfss_wire_bonds = True,
         lead = dict(start_straight=0.500, end_straight=0.500)))

gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
"""
Set 3 separated fluxonia with rectangles. Make sure the layers of fluxonium components are correct for fabrication process. 
"""
x_center, y_center = 0.000, 0.000
x_dis, y_dis = 1.350, 0.000
q_opt = {'pos_x': -x_dis, 'pos_y': 0, 'orientation': 0, 
         'pad_gap': 0.030, 'arm_width': 0.008, 'arm_length': 0.090, 'arm_fillet': 0.020,
         'pad_width': 0.200, 'pad_height': 0.050, 'pad_fillet': 0.050, 
         'pocket_width': 0.350, 'pocket_height': 0.500, 'pocket_fillet': 0.050, 
         'layer': 1}
Q1 = Fillet_Qubit(design, 'Q1', q_opt)

# Remove Q2 (middle qubit)
# q_opt.update(pos_x=0)
# Q2 = Fillet_Qubit(design, 'Q2', q_opt)

q_opt.update(pos_x=x_dis)
Q3 = Fillet_Qubit(design, 'Q3', q_opt)

gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
"""
Set resonators.
"""
res_width, res_gap = 0.015, 0.009

R1_len = 4.000 # ~ 4.2, 4.3, 4.4
spacing, fillet = 0.700, 0.050 # fillet > 1.5*(res_width+res_gap*2)
l_couple, grd_gap = 0.500, 0.004
R_ext, shift_x, shift_y = 0.200, 0, 0.309
tee_width, tee_length, tee_fillet = 0.050, 0.100, 0.005

Tee_1 = Round_Tee(design, "Tee_1",
    dict(ref_qc = Q1, rel_ori = "90", shift_x = shift_x, shift_y = shift_y,
         cpw_width = res_width, cpw_gap = res_gap, ext_len = R_ext, 
         tee_width = tee_width, tee_length = tee_length, tee_fillet = tee_fillet)
)

stg_q1 = ShortToGround(design, 'stg_q1',
    {'pos_x': Q1.options.pos_x, 'pos_y': Tee_1.options.shift_y + Tee_1.options.tee_width/2 + Tee_1.options.ext_len, 'orientation': '270'})
stg_r1 = ShortToGround(design, 'stg_r1',
    {'pos_x': Q1.options.pos_x-(l_couple+fillet), 'pos_y': port_L1.options.pos_y-TL_gap-res_gap-(TL_width+res_width)/2-grd_gap, 'orientation': '180'})
start_jog = OrderedDict()
start_jog[0] = ["R", 0.9]
R1 = RouteMeander(design, 'R1', 
    options = dict(total_length = R1_len-R_ext, fillet=fillet, hfss_wire_bonds = True, 
                   lead = dict(start_straight=l_couple+fillet, end_straight=0.3,
                               start_jogged_extension = start_jog), 
                   trace_width = res_width, trace_gap = res_gap,
                   meander = dict(
                       spacing=spacing, 
                       asymmetry=0
                   ),
                   pin_inputs = Dict(start_pin = Dict(component = 'stg_r1', pin = 'short'),
                                       end_pin = Dict(component = 'stg_q1', pin = 'short'))))

R2_len = 4.050 # ~ 4.2, 4.3, 4.4
spacing, fillet = 0.700, 0.050 # fillet > 1.5*(res_width+res_gap*2)
l_couple, grd_gap = 0.505, 0.004
R_ext, shift_x, shift_y = 0.200, 0, 0.309
tee_width, tee_length, tee_fillet = 0.050, 0.100, 0.005


start_jog = OrderedDict()
start_jog[0] = ["R", 0.8]
R2 = RouteMeander(design, 'R2', 
    options = dict(total_length = R2_len-R_ext, fillet=fillet, hfss_wire_bonds = True, 
                   lead = dict(start_straight=l_couple+fillet, end_straight=0.3,
                               start_jogged_extension = start_jog), 
                   trace_width = res_width, trace_gap = res_gap,
                   meander = dict(
                       spacing=spacing, 
                       asymmetry=0
                   ),
                   pin_inputs = Dict(start_pin = Dict(component = 'stg_r2', pin = 'short'),
                                       end_pin = Dict(component = 'stg_q2', pin = 'short'))))

R3_len = 4.100 # ~ 4.2, 4.3, 4.4
spacing, fillet = 0.700, 0.050 # fillet > 1.5*(res_width+res_gap*2)
l_couple, grd_gap = 0.510, 0.004
R_ext, shift_x, shift_y = 0.200, 0, 0.309
tee_width, tee_length, tee_fillet = 0.050, 0.100, 0.005

Tee_3 = Round_Tee(design, "Tee_3",
    dict(ref_qc = Q3, rel_ori = "90", shift_x = shift_x, shift_y = shift_y,
         cpw_width = res_width, cpw_gap = res_gap, ext_len = R_ext, 
         tee_width = tee_width, tee_length = tee_length, tee_fillet = tee_fillet)
)

stg_q3 = ShortToGround(design, 'stg_q3',
    {'pos_x': Q3.options.pos_x, 'pos_y': Tee_3.options.shift_y + Tee_3.options.tee_width/2 + Tee_3.options.ext_len, 'orientation': '270'})
stg_r3 = ShortToGround(design, 'stg_r3',
    {'pos_x': Q3.options.pos_x-(l_couple+fillet), 'pos_y': port_L1.options.pos_y-TL_gap-res_gap-(TL_width+res_width)/2-grd_gap, 'orientation': '180'})
start_jog = OrderedDict()
start_jog[0] = ["R", 0.7]
R3 = RouteMeander(design, 'R3', 
    options = dict(total_length = R3_len-R_ext, fillet=fillet, hfss_wire_bonds = True, 
                   lead = dict(start_straight=l_couple+fillet, end_straight=0.3,
                               start_jogged_extension = start_jog), 
                   trace_width = res_width, trace_gap = res_gap,
                   meander = dict(
                       spacing=spacing, 
                       asymmetry=0
                   ),
                   pin_inputs = Dict(start_pin = Dict(component = 'stg_r3', pin = 'short'),
                                       end_pin = Dict(component = 'stg_q3', pin = 'short'))))

gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
"""
Flux bias line for each qubit. 
"""
q_loop_width, q_loop_height, q_loop_trace = 0.060, 0.010, 0.002
loop_H, loop_W, loop_wid = 0.050, 0.090, 0.005
fl_loop_stg_1A = ShortToGround(design, 'fl_loop_stg_1A', 
    dict(pos_x=Q1.options.pos_x-Q1.options.pocket_width/2, pos_y=Q1.options.pos_y-loop_H/2, orientation=180))
fl_loop_stg_1B = ShortToGround(design, 'fl_loop_stg_1B', 
    dict(pos_x=Q1.options.pos_x-Q1.options.pocket_width/2, pos_y=Q1.options.pos_y+loop_H/2, orientation=180))
fl_loop_route_1 = RouteStraight(design, 'fl_loop_route_1',
    dict(trace_width=loop_wid, trace_gap=0, fillet=0.010, hfss_wire_bonds=False,
         lead = dict(start_straight=loop_W, end_straight=loop_W),
         pin_inputs = Dict(start_pin = Dict(component = 'fl_loop_stg_1A', pin = 'short'),
                             end_pin = Dict(component = 'fl_loop_stg_1B', pin = 'short'))))
fl_q_stg_1 = ShortToGround(design, 'fl_q_stg_1', 
    dict(pos_x=Q1.options.pos_x-Q1.options.pocket_width/2, pos_y=Q1.options.pos_y-loop_H/2, orientation=0))
fl_q_route_1 = RoutePathfinder(design, 'fl_q_route_1', 
    dict(trace_width='10um', trace_gap='6um', fillet=0.050, hfss_wire_bonds=True,
         lead = dict(start_straight=0.300, end_straight=0.900),
         pin_inputs = Dict(start_pin = Dict(component = 'port_L3', pin = 'tie'),
                             end_pin = Dict(component = 'fl_q_stg_1', pin = 'short'))))


q_loop_width, q_loop_height, q_loop_trace = 0.060, 0.010, 0.002
loop_H, loop_W, loop_wid = 0.050, 0.090, 0.005
fl_loop_stg_3A = ShortToGround(design, 'fl_loop_stg_3A', 
    dict(pos_x=Q3.options.pos_x+Q3.options.pocket_width/2, pos_y=Q3.options.pos_y-loop_H/2, orientation=  0))
fl_loop_stg_3B = ShortToGround(design, 'fl_loop_stg_3B', 
    dict(pos_x=Q3.options.pos_x+Q3.options.pocket_width/2, pos_y=Q3.options.pos_y+loop_H/2, orientation=  0))
fl_loop_route_3 = RouteStraight(design, 'fl_loop_route_3',
    dict(trace_width=loop_wid, trace_gap=0, fillet=0.010, hfss_wire_bonds=False,
         lead = dict(start_straight=loop_W, end_straight=loop_W),
         pin_inputs = Dict(start_pin = Dict(component = 'fl_loop_stg_3A', pin = 'short'),
                             end_pin = Dict(component = 'fl_loop_stg_3B', pin = 'short'))))
fl_q_stg_3 = ShortToGround(design, 'fl_q_stg_3', 
    dict(pos_x=Q3.options.pos_x+Q3.options.pocket_width/2, pos_y=Q3.options.pos_y-loop_H/2, orientation=180))
fl_q_route_3 = RoutePathfinder(design, 'fl_q_route_3', 
    dict(trace_width='10um', trace_gap='6um', fillet=0.050, hfss_wire_bonds=True,
         lead = dict(start_straight=0.300, end_straight=0.900),
         pin_inputs = Dict(start_pin = Dict(component = 'port_R3', pin = 'tie'),
                             end_pin = Dict(component = 'fl_q_stg_3', pin = 'short'))))

gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
"""
Charge line for each qubit. 
"""
cpw_width, cpw_gap = parse_value(var.cpw_width, var), parse_value(var.cpw_gap, var)
cl_vac, cl_grd, cl_ext = 0.020, 0.010, 0.300
cl_shifty = 0.13
cl_q1_gap_1 = Rectangle(design, 'cl_q1_gap_1', 
    dict(pos_x = Q1.options.pos_x-Q1.options.pocket_width/2-cl_grd-cl_ext/2, pos_y = Q1.options.pos_y+cl_shifty, 
         width = cl_ext, height = cpw_gap*2+cpw_width, subtract=True))
cl_q1_trace_1 = Rectangle(design, 'cl_q1_trace_1', 
    dict(pos_x = Q1.options.pos_x-Q1.options.pocket_width/2-cl_grd-cl_ext/2, pos_y = Q1.options.pos_y+cl_shifty, 
         width = cl_ext-cl_vac*2, height = cpw_width))
cl_q_stg_1 = ShortToGround(design, 'cl_q_stg_1', 
    dict(pos_x = Q1.options.pos_x-Q1.options.pocket_width/2-cl_grd-cl_ext+cl_vac, pos_y = Q1.options.pos_y+cl_shifty, 
         orientation=0))


cl_q_route_1 = RoutePathfinder(design, 'cl_q_route_1', 
    dict(trace_width='10um', trace_gap='6um', fillet=0.050, hfss_wire_bonds=True,
         lead = dict(start_straight=0.200, end_straight=0.100),
         pin_inputs = Dict(start_pin = Dict(component = 'port_L2', pin = 'tie'),
                             end_pin = Dict(component = 'cl_q_stg_1', pin = 'short'))))


cl_q1_gap_3 = Rectangle(design, 'cl_q1_gap_3', 
    dict(pos_x = Q3.options.pos_x+Q3.options.pocket_width/2+cl_grd+cl_ext/2, pos_y = Q3.options.pos_y+cl_shifty, 
         width = cl_ext, height = cpw_gap*2+cpw_width, subtract=True))

cl_q1_trace_3 = Rectangle(design, 'cl_q1_trace_3', 
    dict(pos_x = Q3.options.pos_x+Q3.options.pocket_width/2+cl_grd+cl_ext/2, pos_y = Q3.options.pos_y+cl_shifty, 
         width = cl_ext-cl_vac*2, height = cpw_width))

cl_q_stg_3 = ShortToGround(design, 'cl_q_stg_3', 
    dict(pos_x = Q3.options.pos_x+Q3.options.pocket_width/2+cl_grd+cl_ext-cl_vac, pos_y = Q3.options.pos_y+cl_shifty, 
         orientation=180))

cl_q_route_3 = RoutePathfinder(design, 'cl_q_route_3', 
    dict(trace_width='10um', trace_gap='6um', fillet=0.050, hfss_wire_bonds=True,
         lead = dict(start_straight=0.200, end_straight=0.100),
         pin_inputs = Dict(start_pin = Dict(component = 'port_R2', pin = 'tie'),
                             end_pin = Dict(component = 'cl_q_stg_3', pin = 'short'))))

gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
def find_bc_solutions(L, delta, max_n=100):
    """
    在 n = 1,2,...,max_n 範圍內搜尋 (b, c) 解。
    條件：
      n = (L - delta) / (2c) 為正整數
      floor((L-b)/(2*(b+c))) == n  == (L-delta)/(2c)
    回傳值：一個 list，每項為 (n, c, b_min, b_max)，
      其中 b_min < b <= b_max。
    """
    solutions = []
    for n in range(1, max_n+1):
        # 計算 c
        c = (L - delta) / (2 * n)
        if c <= 0:
            continue
        
        # 計算 b 的不等式邊界
        b_min = ((n+1)*delta - L) / (n*(2*n+3))  # 下界 (strict)
        b_max = delta / (2*n+1)                  # 上界 (inclusive)

        # 判斷是否有可行 b
        if b_min < b_max and b_max>0.1:
            solutions.append((n, c, b_min, b_max))
            print(f"n={n}, c={c:.3f}, b_min={b_min:.3f}, b_max={b_max:.3f}")
    return solutions

In [ ]:
"""
Add folded transmission line between bottom ports (port_L4 and port_R4)
Total length: 49.51576mm with folded structure
"""
# 範例：假設 L=10, delta=4，搜尋 n 到 20
L = 49.61576
delta = 5.8 + 3 - 3.0982456140350876 + 3 - 3.0017236072637754 + 3 - 3.0000302387239244 + 3 - 3.000000530503937
sols = find_bc_solutions(L, delta, max_n=40)
print(sols)
if not sols:
    print("在指定的 n 範圍內未找到可行解。")
else:
    print(f"找到 {len(sols)} 組可行解 (n, c, b_min, b_max)：")
    for n, c, b_min, b_max in sols:
        # 取 b 為上下界中點作示範
        lower_horizontal = b_max
        upper_horizontal = lower_horizontal
        initial_horizontal = lower_horizontal
        vertical_line = c
        lower_vertical = c/2
        upper_vertical = c/2
        n_folds = n
print(f"n={n}, vertical_line={vertical_line:.3f}, lower_horizontal={lower_horizontal:.3f}")
# 傳輸線參數
folded_TL_width, folded_TL_gap = 0.010, 0.006  # 與port_L4, port_R4相同的線寬
total_length = L  # 總長度 49.51576mm

# 計算一個完整折疊單元的長度
one_fold_unit = upper_vertical + upper_horizontal + vertical_line + lower_horizontal + lower_vertical

# 計算需要多少個折疊單元
used_length = initial_horizontal  # 已使用長度
remaining_length = total_length - used_length
#n_folds = int(remaining_length / one_fold_unit)  # 折疊單元數量

# 計算最後橫線長度
final_horizontal = remaining_length - (n_folds * one_fold_unit)

print(f"折疊單元數量: {n_folds}")
print(f"最後橫線長度: {final_horizontal:.3f}mm")

# 產生 folded_path (OrderedDict, value 為 tuple)
folded_path = OrderedDict()
start_x, start_y = port_L4.options.pos_x + 0.2, port_L4.options.pos_y  # 0.2 為起始 lead
current_x, current_y = start_x, start_y
folded_path[0] = (current_x, current_y)
path_index = 1
for i in range(n_folds):
    # 上半部垂直線
    current_y += upper_vertical
    folded_path[path_index] = (current_x, current_y)
    path_index += 1
    # 上橫線 (平的)
    current_x += upper_horizontal
    folded_path[path_index] = (current_x, current_y)
    path_index += 1
    # 垂直線
    current_y -= vertical_line
    folded_path[path_index] = (current_x, current_y)
    path_index += 1
    # 下橫線 (平的)
    current_x += lower_horizontal
    folded_path[path_index] = (current_x, current_y)
    path_index += 1
    # 下半部垂直線
    current_y += lower_vertical
    folded_path[path_index] = (current_x, current_y)
    path_index += 1

# 最後橫線
if final_horizontal > 0:
    current_x += final_horizontal
    folded_path[path_index] = (current_x, current_y)
print(current_x+0.2)

# 使用 RoutePathfinder 來避免 RouteAnchors 的自動連接限制
folded_TL = RouteAnchors(
    design, 'folded_TL',
    dict(
        pin_inputs = dict(
            start_pin = dict(component = 'port_L4', pin = 'tie'),
            end_pin = dict(component = 'port_R4', pin = 'tie')
        ),
        anchors = folded_path,
        trace_width = folded_TL_width,
        trace_gap = folded_TL_gap,
        fillet = 0.01,  # 摺疊處為平角
        hfss_wire_bonds = True,
        lead = dict(start_straight=0.100, end_straight=0.100)
    )
)

gui.rebuild()
gui.autoscale()
gui.screenshot()
print(f"設計的總長度: {total_length}mm")

In [ ]:
def get_xy_on_folded_path(folded_path, s):
    """
    給定folded_path (OrderedDict, key為int, value為(x, y)) 和長度s (mm)，
    回傳走s後的(x, y)座標。
    若s超過總長度，則回傳最後一點。
    """
    import numpy as np
    if s<=0.2:
        return (port_L4.options.pos_x + s, port_L4.options.pos_y)
    if s>=49.61576:
        return (port_R4.options.pos_x - (0.2-(s -49.61576)), port_R4.options.pos_y)
    s=s-0.2
    

    # 取得所有點
    points = list(folded_path.values())
    # 計算每段長度
    seg_lens = [np.hypot(points[i+1][0]-points[i][0], points[i+1][1]-points[i][1]) for i in range(len(points)-1)]
    total = 0
    for i, seg_len in enumerate(seg_lens):
        if total + seg_len >= s:
            # 在這一段內
            remain = s - total
            x0, y0 = points[i]
            x1, y1 = points[i+1]
            ratio = remain / seg_len if seg_len != 0 else 0
            x = x0 + (x1 - x0) * ratio
            y = y0 + (y1 - y0) * ratio
            return (x, y)
        total += seg_len
    # 若超過總長度，回傳最後一點
    return points[-1]

In [ ]:
class LShapedCPW:
    """
    通用的 L 型 CPW 走线类，支持以下六种类型：
      - initial_L: 向下 vertical，再向右剩余 horizontal
      - right_L: 向右 small (0.05)，再向下剩余 (total_length - small)
      - left_L: 向左 small (0.05)，再向下剩余
      - bottom_L: 向下 small (0.2)，再向右剩余
      - top: 向下 entire total_length（直线）
      - end_L: 向下 0.6，再向左剩余
    """
    TYPES = ("initial_L", "right_L", "left_L", "bottom_L", "top", "end_L")

    def __init__(self, design, name,
                 start_x, start_y,
                 total_length,
                 trace_width="10um", trace_gap="6um", fillet=0.01,
                 trace_type=None,
                 folded_path=None, s=None):
        self.design = design
        self.name = name
        self.start_x, self.start_y = get_xy_on_folded_path(folded_path, s)
        self.total = total_length
        self.start_component = ""
        self.trace_width = trace_width
        self.trace_gap = trace_gap
        self.fillet = fillet
        self.start_pin = "bottom"  # 默认起始 pin
        self.folded_path = folded_path
        self.s = s

        # 决定类型
        self.trace_type = trace_type or self._determine_type()
        if self.trace_type not in self.TYPES:
            raise ValueError(f"未知类型 {self.trace_type}")

        # 调用对应的 builder
        getattr(self, f"_build_{self.trace_type}")()

    def _determine_type(self):
        if self.folded_path is None or self.s is None:
            return "initial_L"  # fallback
        if self.s <= 0.2:
            return "initial_L"
        if self.s >= 49.61576:
            return "end_L"
        self.s = self.s - 0.2  # 调整 s，去掉起始的 0.2 mm
        import numpy as np
        points = list(self.folded_path.values())
        seg_lens = [np.hypot(points[i+1][0]-points[i][0], points[i+1][1]-points[i][1]) for i in range(len(points)-1)]
        total = 0
        for i, seg_len in enumerate(seg_lens):
            if total + seg_len >= self.s:
                # i=0: 上半部垂直線
                # i=1: 上橫線
                # i=2: 垂直線
                # i=3: 下橫線
                # i=4: 下半部垂直線
                idx_mod = i % 5
                
                if idx_mod == 0:
                    return "right_L"
                elif idx_mod == 1:
                    return "top"
                elif idx_mod == 2:
                    return "left_L"
                elif idx_mod == 3:
                    return "bottom_L"
                elif idx_mod == 4:
                    return "right_L"
            total += seg_len

    def _build_initial_L(self):
        # 向下 self.vertical，再向右剩余
        down = 0.6
        sx, sy = self.start_x, self.start_y
        short_x = sx + (self.total - down)
        short_y = sy - down
        self._add_short(short_x, short_y, orientation="0")
        #folded_TL.add_pin(f"{self.name}_start", points = [[sx,sy],[sx,sy-0.1]],width = 10e-2)
        VirtualJunction(design, f"{self.name}_start", options=dict(
            pos_x=sx,
            pos_y=sy,
            pin_width='10um'
        ))
        self.start_component = f"{self.name}_start"
        anchors = {
            "0": (sx, short_y)
            #"1": (short_x, short_y)
        }
        self._add_route(anchors)


    def _build_right_L(self):
        # 先向右 small，再向下 total-small
        small = 0.05
        sx, sy = self.start_x, self.start_y
        mid_x = sx + small
        mid_y = sy
        short_x = mid_x
        short_y = sy - (self.total - small)

        folded_TL.add_pin(f"{self.name}_start", points = [[sx,sy],[sx+0.000001,sy]],width = 10e-2)

        # 判斷是否需要再折一次
        port4y = port_R4.options.pos_y
        if short_y < port4y - 0.6:
            fold_x = mid_x
            fold_y = port4y - 0.6
            short_x = fold_x + (self.total - (mid_x - sx +  mid_y-fold_y))
            short_y = fold_y
            
            # 最後 short
            self._add_short(short_x, short_y, orientation="0")
            self.start_component = 'folded_TL'
            self.start_pin = f"{self.name}_start"
            anchors = {
                "0": (mid_x, mid_y),
                "1": (fold_x, fold_y)
            }
            self._add_route(anchors)

            
        else:
            # 原本的兩段
            self._add_short(short_x, short_y, orientation="270")
            

            self.start_component = 'folded_TL'
            self.start_pin = f"{self.name}_start"
            anchors = {
                "0": (mid_x, mid_y),
            }
            self._add_route(anchors)

    def _build_left_L(self):
        # 向左 0.05，再向下剩余
        small = 0.05
        sx, sy = self.start_x, self.start_y
        mid_x = sx - small
        mid_y = sy
        short_x = mid_x
        short_y = sy - (self.total - small)
        self._add_short(short_x, short_y, orientation="270")
        VirtualJunction(design, f"{self.name}_start", options=dict(
            pos_x=sx,
            pos_y=sy,
            pin_width='10um'
        ))
        self.start_pin = "left"
        VirtualJunction(design, f"{self.name}_mid", options=dict(
            pos_x=mid_x,
            pos_y=mid_y,
            pin_width='10um'
        ))
        # 第一段：向右
        RouteStraight(
            self.design, f"{self.name}_h",
            options=dict(
                pin_inputs=dict(
                    start_pin=dict(component=f"{self.name}_start", pin=self.start_pin),
                    end_pin=dict(component=self.name+"_mid", pin="right")  # 先連到短路終點
                ),
                trace_width=self.trace_width,
                trace_gap=self.trace_gap
            )
        )
        # 第二段：向下
        RouteStraight(
            self.design, f"{self.name}_v",
            options=dict(
                pin_inputs=dict(
                    start_pin=dict(component=self.name+"_mid", pin="bottom"),
                    end_pin=dict(component=self.short_name, pin="short")
                ),
                trace_width=self.trace_width,
                trace_gap=self.trace_gap
            )
        )

    def _build_bottom_L(self):
        # 向下 0.2，再向右剩余
        down = 0.2
        sx, sy = self.start_x, self.start_y
        mid_x = sx
        mid_y = sy - down
        short_x = sx + (self.total - down)
        short_y = sy - down
        self._add_short(short_x, short_y, orientation="0")
        VirtualJunction(design, f"{self.name}_start", options=dict(
            pos_x=sx,
            pos_y=sy,
            pin_width='10um'
        ))
        VirtualJunction(design, f"{self.name}_mid", options=dict(
            pos_x=mid_x,
            pos_y=mid_y,
            pin_width='10um'
        ))
        # 第一段：向右
        RouteStraight(
            self.design, f"{self.name}_h",
            options=dict(
                pin_inputs=dict(
                    start_pin=dict(component=f"{self.name}_start", pin=self.start_pin),
                    end_pin=dict(component=self.name+"_mid", pin="bottom")  # 先連到短路終點
                ),
                trace_width=self.trace_width,
                trace_gap=self.trace_gap
            )
        )
        # 第二段：向下
        RouteStraight(
            self.design, f"{self.name}_v",
            options=dict(
                pin_inputs=dict(
                    start_pin=dict(component=self.name+"_mid", pin="right"),
                    end_pin=dict(component=self.short_name, pin="short")
                ),
                trace_width=self.trace_width,
                trace_gap=self.trace_gap
            )
        )

    def _build_top(self):
        # 向下 entire total_length（直线）
        sx, sy = self.start_x, self.start_y
        short_x = sx
        short_y = sy - self.total
        self._add_short(short_x, short_y, orientation="270")
        VirtualJunction(design, f"{self.name}_start", options=dict(
            pos_x=sx,
            pos_y=sy,
            pin_width='10um'
        ))
        RouteStraight(design, self.name, options=dict(
            pin_inputs=dict(
                start_pin=dict(component=f"{self.name}_start", pin="bottom"),
                end_pin=dict(component=self.short_name, pin="short")
            ),
            trace_width='10um', trace_gap='6um'
        ))

    def _build_end_L(self):
        down = 0.8
        sx, sy = self.start_x, self.start_y
        mid_x = sx
        mid_y = sy - down
        short_x = sx - (self.total - down)
        short_y = mid_y
        self._add_short(short_x, short_y, orientation="180")
        folded_TL.add_pin(f"{self.name}_start", points = [[sx,sy],[sx,sy-0.000001]],width = 10e-2)
        
        self.start_component = 'folded_TL'
        self.start_pin = f"{self.name}_start"
        anchors = {
            "0": (mid_x, mid_y),
        }
        self._add_route(anchors)

    def _add_short(self, x, y, orientation):
        stg_name = f"{self.name}_stg"
        ShortToGround(self.design, stg_name,
                      options=dict(
                          pos_x=x,
                          pos_y=y,
                          orientation=orientation
                      ))
        self.short_name = stg_name

    def _add_route(self, anchors):
        RouteAnchors(self.design, self.name,
            options=dict(
                pin_inputs=dict(
                    start_pin=dict(component=self.start_component, pin=self.start_pin),
                    end_pin=dict(component=self.short_name, pin="short")
                ),
                trace_width=self.trace_width,
                trace_gap=self.trace_gap,
                fillet=self.fillet,
                anchors=anchors
            )
        )


In [ ]:
# 參數設定
cpw_p1_total_length = 2.06001   # mm
cpw_p1_vertical = 0.6           # mm

s_list =[0.1, 9.07283, 19.48473, 30.13103, 40.54293, 49.71576]
#s_list =[0.1, 0.65, 19.48473, 31, 40.45, 49.71576]
length_p = [2.06001, 0.543278, 0.298913, 0.298913, 0.543278, 2.06001]  # 每段的长度

for idx, s_val in enumerate(s_list):
    LShapedCPW(design, f"cpw_p{idx}",
           start_x, start_y,
           total_length=length_p[idx],folded_path=folded_path, s=s_val)

gui.rebuild()
gui.autoscale()
gui.screenshot()


In [ ]:
class UShapeComponent:
    """
    U形元件類別
    
    參數說明:
    - cx, cy: 中心點座標
    - bl: 底部寬度 (預設 0.3)
    - sh: 側邊高度 (預設 0.1)  
    - tw: 線條厚度 (預設 0.01)
    """
    
    def __init__(self, design, base_name='u_shape', bl=0.3, sh=0.1, tw=0.01):
        """
        初始化 U 形元件
        
        Args:
            design: 設計物件
            base_name: 元件基礎名稱
            bl: 底部寬度
            sh: 側邊高度
            tw: 線條厚度
        """
        self.design = design
        self.base_name = base_name
        self.bl = bl  # 底部寬度
        self.sh = sh  # 側邊高度
        self.tw = tw  # 線條厚度
        
        # 儲存矩形元件的參考
        self.rect_bottom = None
        self.rect_left = None
        self.rect_right = None
    
    def create(self, cx, cy):
        """
        在指定位置創建 U 形元件
        
        Args:
            cx: 中心點 X 座標
            cy: 中心點 Y 座標
        """
        # === 計算各部分位置與尺寸 ===
        
        # 底部矩形
        self.rect_bottom = Rectangle(
            self.design, 
            f'{self.base_name}_bottom', 
            options=dict(
                pos_x=cx,
                pos_y=cy,
                width=self.bl,
                height=self.tw
            )
        )
        
        # 左側矩形
        left_x = cx - self.bl / 2 + self.tw / 2
        left_y = cy + self.sh / 2 - self.tw / 2
        self.rect_left = Rectangle(
            self.design,
            f'{self.base_name}_left',
            options=dict(
                pos_x=left_x,
                pos_y=left_y,
                width=self.tw,
                height=self.sh
            )
        )
        
        # 右側矩形
        right_x = cx + self.bl / 2 - self.tw / 2
        right_y = cy + self.sh / 2 - self.tw / 2
        self.rect_right = Rectangle(
            self.design,
            f'{self.base_name}_right',
            options=dict(
                pos_x=right_x,
                pos_y=right_y,
                width=self.tw,
                height=self.sh
            )
        )

        VirtualJunction(design, f"{self.base_name}_node", options=dict(
            pos_x=cx,
            pos_y=cy,
            pin_width='10um'
        ))
        
        return self
    
    def update_position(self, cx, cy):
        """
        更新 U 形元件位置
        
        Args:
            cx: 新的中心點 X 座標
            cy: 新的中心點 Y 座標
        """
        if not self.rect_bottom:
            raise ValueError("請先使用 create() 方法創建元件")
        
        # 更新底部矩形
        self.rect_bottom.options['pos_x'] = cx
        self.rect_bottom.options['pos_y'] = cy
        
        # 更新左側矩形
        left_x = cx - self.bl / 2 + self.tw / 2
        left_y = cy + self.sh / 2 - self.tw / 2
        self.rect_left.options['pos_x'] = left_x
        self.rect_left.options['pos_y'] = left_y
        
        # 更新右側矩形
        right_x = cx + self.bl / 2 - self.tw / 2
        right_y = cy + self.sh / 2 - self.tw / 2
        self.rect_right.options['pos_x'] = right_x
        self.rect_right.options['pos_y'] = right_y
    
    def update_dimensions(self, bl=None, sh=None, tw=None):
        """
        更新 U 形元件尺寸
        
        Args:
            bl: 底部寬度 (可選)
            sh: 側邊高度 (可選)
            tw: 線條厚度 (可選)
        """
        if not self.rect_bottom:
            raise ValueError("請先使用 create() 方法創建元件")
        
        # 更新參數
        if bl is not None:
            self.bl = bl
        if sh is not None:
            self.sh = sh
        if tw is not None:
            self.tw = tw
        
        # 獲取當前中心位置
        cx = self.rect_bottom.options['pos_x']
        cy = self.rect_bottom.options['pos_y']
        
        # 重新計算位置和尺寸
        self.update_position(cx, cy)
        
        # 更新尺寸
        self.rect_bottom.options['width'] = self.bl
        self.rect_bottom.options['height'] = self.tw
        
        self.rect_left.options['width'] = self.tw
        self.rect_left.options['height'] = self.sh
        
        self.rect_right.options['width'] = self.tw
        self.rect_right.options['height'] = self.sh
    
    def get_components(self):
        """
        取得所有矩形元件
        
        Returns:
            tuple: (底部矩形, 左側矩形, 右側矩形)
        """
        return (self.rect_bottom, self.rect_left, self.rect_right)
    
    def delete(self):
        """
        刪除 U 形元件的所有矩形
        """
        components = [self.rect_bottom, self.rect_left, self.rect_right]
        for component in components:
            if component:
                try:
                    component.delete()
                except:
                    pass
        
        self.rect_bottom = None
        self.rect_left = None
        self.rect_right = None


# === 使用範例 ===
def example_usage():
    """
    使用範例
    """
    # 創建 U 形元件
    u_shape = UShapeComponent(design, 'my_u_shape')
    
    # 在指定位置創建
    cx, cy = 0, 0  # 輸入你的座標
    u_shape.create(cx, cy)
    
    # 重建和顯示
    gui.rebuild()
    gui.autoscale()
    gui.screenshot()
    
    # 可選：更新位置
    # u_shape.update_position(1, 1)
    
    # 可選：更新尺寸
    # u_shape.update_dimensions(bl=0.5, sh=0.2, tw=0.02)
    
    return u_shape


# === 快速創建函數 ===
def create_u_shape(design, cx, cy, name='u_shape', bl=0.3, sh=0.1, tw=0.01):
    """
    快速創建 U 形元件的便利函數
    
    Args:
        design: 設計物件
        cx, cy: 中心座標
        name: 元件名稱
        bl: 底部寬度
        sh: 側邊高度  
        tw: 線條厚度
    
    Returns:
        UShapeComponent: U 形元件實例
    """
    u_shape = UShapeComponent(design, name, bl, sh, tw)
    u_shape.create(cx, cy)
    return u_shape




qubit_couple_gap = 0.06
padbottom_x = Q1.options.pos_x
padbottom_y = Q1.options.pos_y - Q1.options.pad_gap/2 - Q1.options.pad_height -Q1.options.arm_length - 0.005 - qubit_couple_gap
coupling_pad1 = create_u_shape(design, cx=padbottom_x, cy=padbottom_y, name='coupling_pad1')   

padbottom_x = Q3.options.pos_x
padbottom_y = Q3.options.pos_y - Q3.options.pad_gap/2 - Q3.options.pad_height -Q3.options.arm_length - 0.005 - qubit_couple_gap
coupling_pad3 = create_u_shape(design, cx=padbottom_x, cy=padbottom_y, name='coupling_pad3')  

    # 重建和顯示
gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
for i in range(n_folds):
    idx_start = 1 + i*5
    idx_end = 2 + i*5
    if idx_end >= len(folded_path):
        break
    x0, y0 = folded_path[idx_start]
    x1, y1 = folded_path[idx_end]
    xm = (x0 + x1) / 2
    ym = (y0 + y1) / 2
    # 放 VirtualJunction
    VirtualJunction(design, f"couple_point{i+1}", options=dict(
        pos_x=xm,
        pos_y=ym,
        pin_width='10um'
    ))


RoutePathfinder(
    design, 'cpw_couple1',
    dict(
        trace_width='10um',
        trace_gap='6um',
        fillet=0.01,
        hfss_wire_bonds=True,
        lead=dict(start_straight=0.1, end_straight=0.1),
        pin_inputs=dict(
            start_pin=dict(component='couple_point15', pin='right'),
            end_pin=dict(component='coupling_pad1_node', pin='left')
        )
    )
)

RoutePathfinder(
    design, 'cpw_couple2',
    dict(
        trace_width='10um',
        trace_gap='6um',
        fillet=0.01,
        hfss_wire_bonds=True,
        lead=dict(start_straight=0.1, end_straight=0.1),
        pin_inputs=dict(
            start_pin=dict(component='couple_point20', pin='right'),
            end_pin=dict(component='coupling_pad3_node', pin='left')
        )
    )
)

gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
x_mark, y_mark = 2.5, 4.1
mark_pocket_1 = Rectangle(design, 'mark_pocket_1',
    dict(pos_x= x_mark, pos_y= y_mark, width=0.100, height=0.100, subtract=True))
mark_cross_1 = Cross(design, 'mark_cross_1', 
    dict(pos_x= x_mark, pos_y= y_mark, layer='2', width=0.040, height=0.040, trace_width=0.005, subtract=True))
mark_quarter_1 = Rectangle(design, 'mark_quarter_1',
    dict(pos_x= x_mark+0.0175, pos_y= y_mark+0.0175, layer='2', width=0.005, height=0.005, subtract=True))

mark_2 = design.copy_multiple_qcomponents(
    [mark_pocket_1, mark_cross_1, mark_quarter_1], ['mark_pocket_2', 'mark_cross_2', 'mark_quarter_2'],
    [dict(pos_x=-x_mark, pos_y= y_mark), dict(pos_x=-x_mark, pos_y= y_mark), dict(pos_x=-(x_mark+0.0175), pos_y= y_mark+0.0175)])
mark_3 = design.copy_multiple_qcomponents(
    [mark_pocket_1, mark_cross_1, mark_quarter_1], ['mark_pocket_3', 'mark_cross_3', 'mark_quarter_3'],
    [dict(pos_x=-x_mark, pos_y=-y_mark), dict(pos_x=-x_mark, pos_y=-y_mark), dict(pos_x=-(x_mark+0.0175), pos_y= -(y_mark+0.0175))])
mark_4 = design.copy_multiple_qcomponents(
    [mark_pocket_1, mark_cross_1, mark_quarter_1], ['mark_pocket_4', 'mark_cross_4', 'mark_quarter_4'],
    [dict(pos_x= x_mark, pos_y=-y_mark), dict(pos_x= x_mark, pos_y=-y_mark), dict(pos_x= x_mark+0.0175, pos_y= -(y_mark+0.0175))])

gui.rebuild()
gui.autoscale()
gui.screenshot()

In [ ]:
design.components.keys()

# To GDS

In [ ]:
gds_render = design.renderers.gds
gds_render.options

In [ ]:
# gds_render.options.update(dict(path_filename='fluxonium_JJ_sample_wide_loop.gds', junction_pad_overlap='0um'))

In [ ]:
gds_render.options.fabricate = True
gds_render.options.negative_mask = dict(main=[2])
gds_render.options.no_cheese['view_in_file']['main']={1: False, 2: False}
gds_render.options.no_cheese['buffer']='20um'
gds_render.options.cheese['edge_nocheese']='0.9mm'
gds_render.options.cheese['view_in_file']['main']={1: True, 2:False}
gds_render.options.cheese.update(dict(cheese_0_x=0.005, cheese_0_y=0.005, delta_x=0.05, delta_y=0.05))

In [ ]:
design.qgeometry.tables['junction']

In [ ]:
design.renderers.gds.export_to_gds('Fluxonium_readout_cl_fl_filtter.gds')